## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt

## Import Datasets

In [2]:
#Load Datasets
fake_df = pd.read_parquet("/Volumes/E$/CEIR/Clean Dumps/Fake/fake_2025-08.parquet") # Update File name as needed
genuine_df = pd.read_parquet("/Volumes/E$/CEIR/Clean Dumps/Genuine/genuine_2025-08.parquet") # Update File name as needed

In [3]:
fake_df.head()

,imei_first_seen,last_seen,imei,imei_status,imsi,msisdn,rat,cgi,id_type,id_number,prefix,mno,gender,birth_year,age,district,mcc,country
0,2025-08-20 04:23:41,1970-01-01 00:00:00,00000000003569,,,,,,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,,UNKNOWN
1,2025-08-31 21:42:07,2026-02-23 13:24:45,00000002567439,W,641010414941010,256741849896,,,NATIONAL_ID,CM0404910H55RH,074,AIRTEL,Male,2004,22,MAYUGE,641,Uganda
2,2025-08-31 21:08:48,2026-02-25 18:37:40,00000209100247,W,641010246282271,256707302482,,,NaN,NaN,NaN,AIRTEL,NaN,<NA>,<NA>,NaN,641,Uganda
3,2025-08-31 21:07:52,2025-12-13 19:44:45,00000354123351,W,641010404937144,,,,NaN,NaN,NaN,AIRTEL,NaN,<NA>,<NA>,NaN,641,Uganda
4,2025-08-14 17:41:21,1970-01-01 00:00:00,00100030040006,,,,,,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,,UNKNOWN


In [4]:
genuine_df.head()

,imei_first_seen,last_seen,tac,imei,imei_status,imsi,msisdn,rat,cgi,oem,...,id_type,id_number,prefix,mno,gender,birth_year,age,district,mcc,country
0,2025-08-08 06:04:13,2026-01-04 10:34:48,00440111,00440111560923,W,641010416031310,256754938113,,,Not Known,...,NaN,NaN,NaN,AIRTEL,NaN,<NA>,<NA>,NaN,641,Uganda
1,2025-08-31 21:29:10,2026-03-08 22:44:16,00440111,00440111638671,W,641010275185255,,,,Not Known,...,NaN,NaN,NaN,AIRTEL,NaN,<NA>,<NA>,NaN,641,Uganda
2,2025-08-31 21:40:02,2025-11-13 20:54:42,00440111,00440111684336,W,641010271823428,,,,Not Known,...,NaN,NaN,NaN,AIRTEL,NaN,<NA>,<NA>,NaN,641,Uganda
3,2025-08-08 10:45:47,2025-10-17 11:18:45,00440172,00440172565947,W,641010270402476,,,,Not Known,...,NaN,NaN,NaN,AIRTEL,NaN,<NA>,<NA>,NaN,641,Uganda
4,2025-08-08 05:58:53,1970-01-01 00:00:00,00440245,00440245567644,,,,,,Not Known,...,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,,UNKNOWN


## Fake IMEI - Analysis

In [16]:

# FEATURE PREP — derive time parts + age brackets from raw data
# ============================================================
def prep_time_and_age(fake_df, time_col='imei_first_seen'):
    # Ensure the timestamp column is real datetime before using .dt
    fake_df[time_col] = pd.to_datetime(fake_df[time_col], errors='coerce')

    # Time dimensions used across the monthly reports
    fake_df['year_month'] = fake_df[time_col].dt.to_period('M')
    fake_df['hour']       = fake_df[time_col].dt.hour
    fake_df['dow']        = fake_df[time_col].dt.day_name()

    # Age -> ordered bracket. right=True means each bin is (low, high];
    # include_lowest=True so an exact age of 0 still lands in "<18".
    bins   = [0, 17, 30, 40, 50, 60, 70, 100, 120]
    labels = ["<18", "18-30", "31-40", "41-50", "51-60", "61-70", "71-100", "100-120"]
    numeric_age = pd.to_numeric(fake_df['age'], errors='coerce')
    age_bracket = pd.cut(numeric_age, bins=bins, labels=labels,
                         right=True, include_lowest=True)

    # Missing / out-of-range ages -> "Roamers" (no age data on file)
    age_bracket = age_bracket.astype('object').fillna('unknown')
    fake_df['age_bracket'] = pd.Categorical(age_bracket,
                                            categories=labels + ['unknown'],
                                            ordered=True)
    return fake_df


# REPORT HELPER — one month-by-dimension crosstab + display
# ============================================================
def report_by_month(df, col, title, fill=None, dropna=True, margins_name='TOTAL'):
    series = df[col]
    # Fill nulls when a label is given (cast to object first so it works
    # even if the column is categorical, e.g. mno/gender)
    if fill is not None:
        series = series.astype('object').fillna(fill)

    tab = pd.crosstab(df['year_month'], series, dropna=dropna,
                      margins=True, margins_name=margins_name)
    print(f"\n{title}\n{'='*70}")
    display(tab)
    return tab


# RUN — build features, then output the four monthly tables
# ============================================================
fake_df = prep_time_and_age(fake_df)

mno_tab      = report_by_month(fake_df, 'mno',         'Fake per MNO by month',       fill='unknown')
gender_tab   = report_by_month(fake_df, 'gender',      'Fake per gender by month',    fill='unknown')
age_tab      = report_by_month(fake_df, 'age_bracket', 'Fake per age group by month', dropna=False)
district_tab = report_by_month(fake_df, 'district',    'Fake per district by month')
country_tab  = report_by_month(fake_df, 'country',     'Fake per country by month',   fill='unknown')


Fake per MNO by month


mno,AIRTEL,MTN,unknown,TOTAL
year_month,,,,
2025-08,67891,35768,33123,136782
TOTAL,67891,35768,33123,136782



Fake per gender by month


gender,Female,Male,Undefined,unknown,TOTAL
year_month,,,,,
2025-08,34257,34057,1,68467,136782
TOTAL,34257,34057,1,68467,136782



Fake per age group by month


age_bracket,<18,18-30,31-40,41-50,51-60,61-70,71-100,100-120,unknown,TOTAL
year_month,,,,,,,,,,
2025-08,2,23128,21122,12779,7118,2908,1258,0,68467,136782
TOTAL,2,23128,21122,12779,7118,2908,1258,0,68467,136782



Fake per district by month


district,ABIM,ADJUMANI,AGAGO,ALEBTONG,AMOLATAR,AMUDAT,AMURIA,AMURU,APAC,ARUA,...,SHEEMA,SIRONKO,SOROTI,SSEMBABULE,TORORO,UNKNOWN,WAKISO,YUMBE,ZOMBO,TOTAL
year_month,,,,,,,,,,,,,,,,,,,,,
2025-08,37,196,107,186,96,135,201,173,203,952,...,661,876,224,559,1295,211,3344,323,433,68315
TOTAL,37,196,107,186,96,135,201,173,203,952,...,661,876,224,559,1295,211,3344,323,433,68315



Fake per country by month


country,Belgium,Congo,Democratic Republic of Congo,Guam,International Networks,Kenya,Netherlands,Nigeria,Oman,Rwanda,South Sudan,Tanzania,UNKNOWN,Uganda,United Arab Emirates,United Kingdom,Zambia,TOTAL
year_month,,,,,,,,,,,,,,,,,,
2025-08,1,1,778,3,6,903,4,1,1,232,88,22,31044,103659,8,30,1,136782
TOTAL,1,1,778,3,6,903,4,1,1,232,88,22,31044,103659,8,30,1,136782


## Genuine IMEI - Analysis

In [17]:

# FEATURE PREP — derive time parts + age brackets from raw data
# ============================================================
def prep_time_and_age(genuine_df, time_col='imei_first_seen'):
    # Ensure the timestamp column is real datetime before using .dt
    genuine_df[time_col] = pd.to_datetime(genuine_df[time_col], errors='coerce')

    # Time dimensions used across the monthly reports
    genuine_df['year_month'] = genuine_df[time_col].dt.to_period('M')
    genuine_df['hour']       = genuine_df[time_col].dt.hour
    genuine_df['dow']        = genuine_df[time_col].dt.day_name()

    # Age -> ordered bracket. right=True means each bin is (low, high];
    # include_lowest=True so an exact age of 0 still lands in "<18".
    bins   = [0, 17, 30, 40, 50, 60, 70, 100, 120]
    labels = ["<18", "18-30", "31-40", "41-50", "51-60", "61-70", "71-100", "100-120"]
    numeric_age = pd.to_numeric(genuine_df['age'], errors='coerce')
    age_bracket = pd.cut(numeric_age, bins=bins, labels=labels,
                         right=True, include_lowest=True)

    # Missing / out-of-range ages -> "Roamers" (no age data on file)
    age_bracket = age_bracket.astype('object').fillna('unknown')
    genuine_df['age_bracket'] = pd.Categorical(age_bracket,
                                                categories=labels + ['unknown'],
                                                ordered=True)
    return genuine_df


# REPORT HELPER — one month-by-dimension crosstab + display
# ============================================================
def report_by_month(df, col, title, fill=None, dropna=True, margins_name='TOTAL'):
    series = df[col]
    # Fill nulls when a label is given (cast to object first so it works
    # even if the column is categorical, e.g. mno/gender)
    if fill is not None:
        series = series.astype('object').fillna(fill)

    tab = pd.crosstab(df['year_month'], series, dropna=dropna,
                      margins=True, margins_name=margins_name)
    print(f"\n{title}\n{'='*70}")
    display(tab)
    return tab


# RUN — build features, then output the four monthly tables
# ============================================================
genuine_df = prep_time_and_age(genuine_df)

mno_tab      = report_by_month(genuine_df, 'mno',         'Genuine per MNO by month',       fill='unknown')
gender_tab   = report_by_month(genuine_df, 'gender',      'Genuine per gender by month',    fill='unknown')
age_tab      = report_by_month(genuine_df, 'age_bracket', 'Genuine per age group by month', dropna=False)
district_tab = report_by_month(genuine_df, 'district',    'Genuine per district by month')
country_tab  = report_by_month(genuine_df, 'country',     'Genuine per country by month',   fill='unknown')


Genuine per MNO by month


mno,AIRTEL,HAMILTON,MTN,unknown,TOTAL
year_month,,,,,
2025-08,1372939,2,561416,1933839,3868196
TOTAL,1372939,2,561416,1933839,3868196



Genuine per gender by month


gender,Female,Male,Undefined,unknown,TOTAL
year_month,,,,,
2025-08,697081,697000,8,2474107,3868196
TOTAL,697081,697000,8,2474107,3868196



Genuine per age group by month


age_bracket,<18,18-30,31-40,41-50,51-60,61-70,71-100,100-120,unknown,TOTAL
year_month,,,,,,,,,,
2025-08,145,383221,430453,292402,176148,76883,34823,0,2474121,3868196
TOTAL,145,383221,430453,292402,176148,76883,34823,0,2474121,3868196



Genuine per district by month


district,ABIM,ADJUMANI,AGAGO,ALEBTONG,AMOLATAR,AMUDAT,AMURIA,AMURU,APAC,ARUA,...,SHEEMA,SIRONKO,SOROTI,SSEMBABULE,TORORO,UNKNOWN,WAKISO,YUMBE,ZOMBO,TOTAL
year_month,,,,,,,,,,,,,,,,,,,,,
2025-08,806,3604,1991,2898,1846,10749,4597,4106,3741,19216,...,11033,21251,5912,11694,37369,5161,40467,5889,11545,1394091
TOTAL,806,3604,1991,2898,1846,10749,4597,4106,3741,19216,...,11033,21251,5912,11694,37369,5161,40467,5889,11545,1394091



Genuine per country by month


country,Angola,Australia,Austria,Bahrain,Belgium,Botswana,Burkina Faso,Cameroon,Canada,Central African Republic,...,Turkiye,UNKNOWN,Uganda,Ukraine,United Arab Emirates,United Kingdom,Vietnam,Zambia,Zimbabwe,TOTAL
year_month,,,,,,,,,,,,,,,,,,,,,
2025-08,4,2,16,12,22302,1,3,3,25,1,...,55,1426312,1934357,2,1107,24188,6,176,2,3868196
TOTAL,4,2,16,12,22302,1,3,3,25,1,...,55,1426312,1934357,2,1107,24188,6,176,2,3868196
